# SPR 2026 - Classificação Hierárquica (Cascata) com BERTimbau

**Estratégia:** Classificação em cascata (benigno vs alterado)

## Arquitetura Hierárquica

| Modelo | Tarefa | Classes |
|--------|--------|----------|
| Modelo 1 | Benigno vs Alterado (binário) | 0=Benigno, 1=Alterado |
| Modelo 2 | Classificar Benignos | 1, 2, 3 (Negativo, Benigno, Prob. Benigno) |
| Modelo 3 | Classificar Alterados | 0, 4, 5, 6 (Incompleto, Suspeito, Alt. Sugestivo, Malignidade) |

## Mapeamento BI-RADS

**BENIGNO (1, 2, 3):**
- 1: Negativo (nenhuma alteração)
- 2: Benigno (achado benigno)
- 3: Provavelmente Benigno

**ALTERADO (0, 4, 5, 6):**
- 0: Incompleto (necessita avaliação adicional)
- 4: Suspeito
- 5: Altamente Sugestivo de Malignidade
- 6: Malignidade Comprovada

---
## 📥 MODELO - Baixar com: `models/download_bertimbau_large.ipynb`

**Kaggle Models:** Add Input → Models → `bertimbau-ptbr-complete`

---
**CONFIGURAÇÃO KAGGLE:**
1. Settings → Internet → **OFF**
2. Settings → Accelerator → **GPU T4 x2**
---

In [ ]:
# ===== SPR 2026 - CASCATA BERTIMBAU v1 =====

print("[1/10] Configurando ambiente...")
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# ========== CONFIGURAÇÕES GLOBAIS ==========
SEED = 42
MAX_LEN = 256
BATCH_SIZE = 8
EPOCHS = 5
LR = 2e-5
FOCAL_GAMMA = 2.0
FOCAL_ALPHA = 0.25

# Mapeamento de classes
CLASSES_BENIGNO = [1, 2, 3]  # Negativo, Benigno, Prob. Benigno
CLASSES_ALTERADO = [0, 4, 5, 6]  # Incompleto, Suspeito, Alt. Sugestivo, Malignidade

BIRADS_LABELS = {
    0: 'Incompleto', 1: 'Negativo', 2: 'Benigno', 3: 'Provavelmente Benigno',
    4: 'Suspeito', 5: 'Altamente Sugestivo', 6: 'Malignidade Comprovada'
}

# =============================================

DATA_DIR = '/kaggle/input/competitions/spr-2026-mammography-report-classification'

# Verificar dataset
if not os.path.exists(DATA_DIR):
    DATA_DIR = '/kaggle/input/spr-2026-mammography-report-classification'
    if not os.path.exists(DATA_DIR):
        print("\n" + "="*60)
        print("ERRO: Dataset não encontrado!")
        print("="*60)
        print("\nAdicione o dataset:")
        print("Add Input → Competition → spr-2026-mammography-report-classification")
        raise FileNotFoundError(f"Dataset não encontrado")
print(f"Dataset: {DATA_DIR}")

# Auto-detectar modelo BERTimbau
def find_model_path():
    base = '/kaggle/input'
    def has_config(path):
        return os.path.isdir(path) and os.path.exists(os.path.join(path, 'config.json'))
    def search_dir(directory, depth=0, max_depth=10):
        if depth > max_depth:
            return None
        try:
            for item in os.listdir(directory):
                path = os.path.join(directory, item)
                if os.path.isdir(path):
                    if has_config(path):
                        return path
                    result = search_dir(path, depth + 1, max_depth)
                    if result:
                        return result
        except:
            pass
        return None
    return search_dir(base)

MODEL_PATH = find_model_path()
if MODEL_PATH is None:
    raise FileNotFoundError("Adicione o modelo BERTimbau ao notebook!")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Model: {MODEL_PATH}')
torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
# Focal Loss
print("[2/10] Definindo Focal Loss...")

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
    
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss

print(f'Focal Loss: gamma={FOCAL_GAMMA}, alpha={FOCAL_ALPHA}')

In [ ]:
# Dataset class
class TextDataset(Dataset):
    def __init__(self, texts, labels=None, tokenizer=None, max_len=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            max_length=self.max_len,
            padding='max_length',
            return_tensors='pt',
        )
        item = {
            'input_ids': enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
        }
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

In [ ]:
# Carregar dados
print("[3/10] Carregando dados...")
train_df = pd.read_csv(f'{DATA_DIR}/train.csv')
test_df = pd.read_csv(f'{DATA_DIR}/test.csv')
print(f'Train: {train_df.shape}, Test: {test_df.shape}')

# Criar label binária: Benigno(0) vs Alterado(1)
def map_to_binary(target):
    if target in CLASSES_BENIGNO:
        return 0  # Benigno
    else:
        return 1  # Alterado

train_df['target_binary'] = train_df['target'].apply(map_to_binary)

print("\nDistribuição Binária:")
print(f"  Benigno (0):  {(train_df['target_binary'] == 0).sum()}")
print(f"  Alterado (1): {(train_df['target_binary'] == 1).sum()}")

# Separar subsets
train_benigno = train_df[train_df['target'].isin(CLASSES_BENIGNO)].copy()
train_alterado = train_df[train_df['target'].isin(CLASSES_ALTERADO)].copy()

print(f"\nSubset Benigno: {len(train_benigno)} amostras")
print(f"Subset Alterado: {len(train_alterado)} amostras")

In [ ]:
# Função de treinamento
def train_model(train_texts, train_labels, val_texts, val_labels, num_classes, model_name="Modelo"):
    print(f"\n{'='*60}")
    print(f"Treinando {model_name}")
    print(f"Classes: {num_classes}, Amostras: {len(train_texts)}")
    print(f"{'='*60}")
    
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
    
    train_ds = TextDataset(train_texts, train_labels, tokenizer, MAX_LEN)
    val_ds = TextDataset(val_texts, val_labels, tokenizer, MAX_LEN)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
    
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_PATH, num_labels=num_classes, local_files_only=True
    )
    model.to(device)
    
    criterion = FocalLoss(alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)
    
    best_f1 = 0
    best_model_state = None
    
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        for batch in tqdm(train_loader, desc=f'{model_name} - Epoch {epoch+1}/{EPOCHS}'):
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs.logits, labels)
            loss.backward()
            optimizer.step()
            scheduler.step()
            train_loss += loss.item()
        
        # Validação
        model.eval()
        val_preds, val_true = [], []
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                preds = outputs.logits.argmax(dim=1)
                val_preds.extend(preds.cpu().numpy())
                val_true.extend(batch['labels'].numpy())
        
        val_f1 = f1_score(val_true, val_preds, average='macro')
        print(f'{model_name} Epoch {epoch+1}: Loss={train_loss/len(train_loader):.4f}, Val F1={val_f1:.4f}')
        
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_model_state = model.state_dict().copy()
    
    # Carregar melhor modelo
    model.load_state_dict(best_model_state)
    print(f'\n{model_name} - Melhor Val F1: {best_f1:.4f}')
    
    return model, tokenizer, best_f1

In [ ]:
# ========== MODELO 1: Benigno vs Alterado (BINÁRIO) ==========
print("[4/10] Treinando Modelo 1: Benigno vs Alterado...")

train_texts_1, val_texts_1, train_labels_1, val_labels_1 = train_test_split(
    train_df['report'].values, 
    train_df['target_binary'].values,
    test_size=0.1, random_state=SEED, stratify=train_df['target_binary']
)

model_binary, tokenizer, f1_binary = train_model(
    train_texts_1, train_labels_1, 
    val_texts_1, val_labels_1, 
    num_classes=2,
    model_name="Modelo 1 (Benigno vs Alterado)"
)

print(f"\n✅ Modelo 1 treinado - F1: {f1_binary:.4f}")

In [ ]:
# ========== MODELO 2: Classificar Benignos (1, 2, 3) ==========
print("[5/10] Treinando Modelo 2: Classificar Benignos...")

# Mapear labels para 0, 1, 2 (original: 1, 2, 3)
# 1 -> 0, 2 -> 1, 3 -> 2
label_map_benigno = {1: 0, 2: 1, 3: 2}
label_map_benigno_inv = {0: 1, 1: 2, 2: 3}

train_benigno_mapped = train_benigno['target'].map(label_map_benigno).values

train_texts_2, val_texts_2, train_labels_2, val_labels_2 = train_test_split(
    train_benigno['report'].values, 
    train_benigno_mapped,
    test_size=0.1, random_state=SEED, stratify=train_benigno_mapped
)

model_benigno, _, f1_benigno = train_model(
    train_texts_2, train_labels_2, 
    val_texts_2, val_labels_2, 
    num_classes=3,
    model_name="Modelo 2 (Benignos: 1,2,3)"
)

print(f"\n✅ Modelo 2 treinado - F1: {f1_benigno:.4f}")

In [ ]:
# ========== MODELO 3: Classificar Alterados (0, 4, 5, 6) ==========
print("[6/10] Treinando Modelo 3: Classificar Alterados...")

# Mapear labels para 0, 1, 2, 3 (original: 0, 4, 5, 6)
# 0 -> 0, 4 -> 1, 5 -> 2, 6 -> 3
label_map_alterado = {0: 0, 4: 1, 5: 2, 6: 3}
label_map_alterado_inv = {0: 0, 1: 4, 2: 5, 3: 6}

train_alterado_mapped = train_alterado['target'].map(label_map_alterado).values

train_texts_3, val_texts_3, train_labels_3, val_labels_3 = train_test_split(
    train_alterado['report'].values, 
    train_alterado_mapped,
    test_size=0.1, random_state=SEED, stratify=train_alterado_mapped
)

model_alterado, _, f1_alterado = train_model(
    train_texts_3, train_labels_3, 
    val_texts_3, val_labels_3, 
    num_classes=4,
    model_name="Modelo 3 (Alterados: 0,4,5,6)"
)

print(f"\n✅ Modelo 3 treinado - F1: {f1_alterado:.4f}")

In [ ]:
# Resumo do treinamento
print("[7/10] Resumo do treinamento...")
print("="*60)
print("CASCATA - RESUMO DOS MODELOS")
print("="*60)
print(f"\nModelo 1 (Benigno vs Alterado): F1 = {f1_binary:.4f}")
print(f"Modelo 2 (Benignos 1,2,3):       F1 = {f1_benigno:.4f}")
print(f"Modelo 3 (Alterados 0,4,5,6):    F1 = {f1_alterado:.4f}")
print("\n" + "="*60)

In [ ]:
# ========== INFERÊNCIA EM CASCATA ==========
print("[8/10] Inferência em cascata no conjunto de teste...")

# Criar dataset de teste
test_ds = TextDataset(test_df['report'].values, None, tokenizer, MAX_LEN)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)

# Passo 1: Classificação binária (Benigno vs Alterado)
print("\nPasso 1: Classificação Binária")
model_binary.eval()
binary_preds = []
binary_probs = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc='Modelo 1 - Binário'):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model_binary(input_ids=input_ids, attention_mask=attention_mask)
        probs = F.softmax(outputs.logits, dim=1)
        preds = outputs.logits.argmax(dim=1)
        binary_preds.extend(preds.cpu().numpy())
        binary_probs.extend(probs.cpu().numpy())

binary_preds = np.array(binary_preds)
binary_probs = np.array(binary_probs)

print(f"Benigno (0): {(binary_preds == 0).sum()}")
print(f"Alterado (1): {(binary_preds == 1).sum()}")

In [ ]:
# Passo 2: Classificar benignos (1, 2, 3)
print("\nPasso 2: Classificar Benignos")
indices_benigno = np.where(binary_preds == 0)[0]
print(f"Amostras a classificar: {len(indices_benigno)}")

benigno_preds = np.zeros(len(test_df), dtype=int)

if len(indices_benigno) > 0:
    model_benigno.eval()
    
    # Criar subset para benignos
    texts_benigno = test_df['report'].values[indices_benigno]
    ds_benigno = TextDataset(texts_benigno, None, tokenizer, MAX_LEN)
    loader_benigno = DataLoader(ds_benigno, batch_size=BATCH_SIZE)
    
    preds_benigno_temp = []
    with torch.no_grad():
        for batch in tqdm(loader_benigno, desc='Modelo 2 - Benignos'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            outputs = model_benigno(input_ids=input_ids, attention_mask=attention_mask)
            preds = outputs.logits.argmax(dim=1)
            preds_benigno_temp.extend(preds.cpu().numpy())
    
    # Mapear de volta para classes originais (0,1,2 -> 1,2,3)
    preds_benigno_mapped = [label_map_benigno_inv[p] for p in preds_benigno_temp]
    benigno_preds[indices_benigno] = preds_benigno_mapped
    
    print(f"Distribuição benignos:")
    for c in [1, 2, 3]:
        count = sum(1 for p in preds_benigno_mapped if p == c)
        print(f"  Classe {c}: {count}")

In [ ]:
# Passo 3: Classificar alterados (0, 4, 5, 6)
print("\nPasso 3: Classificar Alterados")
indices_alterado = np.where(binary_preds == 1)[0]
print(f"Amostras a classificar: {len(indices_alterado)}")

alterado_preds = np.zeros(len(test_df), dtype=int)

if len(indices_alterado) > 0:
    model_alterado.eval()
    
    # Criar subset para alterados
    texts_alterado = test_df['report'].values[indices_alterado]
    ds_alterado = TextDataset(texts_alterado, None, tokenizer, MAX_LEN)
    loader_alterado = DataLoader(ds_alterado, batch_size=BATCH_SIZE)
    
    preds_alterado_temp = []
    with torch.no_grad():
        for batch in tqdm(loader_alterado, desc='Modelo 3 - Alterados'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            outputs = model_alterado(input_ids=input_ids, attention_mask=attention_mask)
            preds = outputs.logits.argmax(dim=1)
            preds_alterado_temp.extend(preds.cpu().numpy())
    
    # Mapear de volta para classes originais (0,1,2,3 -> 0,4,5,6)
    preds_alterado_mapped = [label_map_alterado_inv[p] for p in preds_alterado_temp]
    alterado_preds[indices_alterado] = preds_alterado_mapped
    
    print(f"Distribuição alterados:")
    for c in [0, 4, 5, 6]:
        count = sum(1 for p in preds_alterado_mapped if p == c)
        print(f"  Classe {c}: {count}")

In [ ]:
# Combinar predições finais
print("[9/10] Combinando predições finais...")

final_predictions = np.zeros(len(test_df), dtype=int)

# Atribuir predições de benignos
final_predictions[indices_benigno] = benigno_preds[indices_benigno]

# Atribuir predições de alterados
final_predictions[indices_alterado] = alterado_preds[indices_alterado]

print("\n" + "="*60)
print("DISTRIBUIÇÃO FINAL DAS PREDIÇÕES")
print("="*60)
for c in range(7):
    count = (final_predictions == c).sum()
    print(f"  Classe {c} ({BIRADS_LABELS[c]:25}): {count:5}")

In [ ]:
# Gerar submissão
print("[10/10] Gerando submissão...")

sample_path = f'{DATA_DIR}/sample_submission.csv'
if os.path.exists(sample_path):
    sample_sub = pd.read_csv(sample_path)
    SUB_ID = sample_sub.columns[0]
    SUB_LABEL = sample_sub.columns[1]
else:
    SUB_ID = 'ID'
    SUB_LABEL = 'target'

submission = pd.DataFrame({SUB_ID: test_df['ID'], SUB_LABEL: final_predictions})
submission.to_csv('/kaggle/working/submission.csv', index=False)

print("\n" + "="*60)
print("CASCATA BERTIMBAU v1 - CONCLUÍDO!")
print("="*60)
print(f"\nModelo 1 (Benigno vs Alterado): F1 = {f1_binary:.4f}")
print(f"Modelo 2 (Benignos 1,2,3):       F1 = {f1_benigno:.4f}")
print(f"Modelo 3 (Alterados 0,4,5,6):    F1 = {f1_alterado:.4f}")

print("\nDistribuição das predições:")
print(submission[SUB_LABEL].value_counts().sort_index())
print("\n✅ submission.csv criado!")